In [20]:
!pip install dagshub optuna imbalanced-learn
!pip install mlflow

In [21]:
#!aws configure
#!aws configure
import dagshub
dagshub.init(repo_owner="rohitbedse",repo_name="yt-comment-sentiment-analysis",mlflow=True)

Initialized MLflow to track repo "rohitbedse/yt-comment-sentiment-analysis"

Repository rohitbedse/yt-comment-sentiment-analysis initialized!

In [22]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow")

In [23]:
# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning With Chnaging Features")

<Experiment: artifact_location='mlflow-artifacts:/54b3eaefe0cb4e4d8246724361152697', creation_time=1775572221042, experiment_id='9', last_update_time=1775572221042, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning With Chnaging Features', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [24]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report,f1_score
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna
import numpy as np

In [25]:
df = pd.read_csv('/content/reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [26]:
# -----------------------------
# STEP 1: Label Mapping
# -----------------------------
df = df.dropna(subset=['category'])
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# -----------------------------
# STEP 2: Show ORIGINAL label distribution
# -----------------------------
print("🔥 ORIGINAL LABELS:")
print(np.unique(df['category'], return_counts=True))

# -----------------------------
# STEP 3: TRAIN-TEST SPLIT (FIRST)
# -----------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# -----------------------------
# STEP 4: TF-IDF (FIT ONLY ON TRAIN)
# -----------------------------
vectorizer = TfidfVectorizer(ngram_range=(1,3), max_features=5000)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# -----------------------------
# STEP 5: Show TRAIN distribution BEFORE SMOTE
# -----------------------------
print("\n📊 TRAIN LABELS BEFORE SMOTE:")
print(np.unique(y_train, return_counts=True))

# -----------------------------
# STEP 6: APPLY SMOTE ONLY ON TRAIN
# -----------------------------
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# -----------------------------
# STEP 7: Show TRAIN distribution AFTER SMOTE
# -----------------------------
print("\n🚀 TRAIN LABELS AFTER SMOTE:")
print(np.unique(y_train_res, return_counts=True))


# -----------------------------
# STEP 8: MLflow logging (ONLY ONCE)
# -----------------------------
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Bigram")
        mlflow.set_tag("experiment", "No_Leakage_Pipeline")

        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("resampling", "SMOTE")
        mlflow.log_param("vectorizer", "TF-IDF Bigram")

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)

        report = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        mlflow.sklearn.log_model(model, f"{model_name}_model")


# -----------------------------
# STEP 9: OPTUNA OBJECTIVE
# -----------------------------
def objective_mnb(trial):
    alpha = trial.suggest_float('alpha', 1e-4, 1.0, log=True)

    model = MultinomialNB(alpha=alpha)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_res, y_train_res,
        test_size=0.2,
        random_state=42,
        stratify=y_train_res
    )

    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)

    return f1_score(y_val, y_pred, average='macro')


# -----------------------------
# STEP 10: RUN OPTUNA
# -----------------------------
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_mnb, n_trials=30)

    print("\n🏆 BEST PARAMS:", study.best_params)

    best_model = MultinomialNB(alpha=study.best_params['alpha'])

    log_mlflow("MultinomialNB", best_model, X_train_res, X_test, y_train_res, y_test)


# -----------------------------
# RUN
# -----------------------------
run_optuna_experiment()

🔥 ORIGINAL LABELS:
(array([0, 1, 2]), array([12644, 15770,  8248]))

📊 TRAIN LABELS BEFORE SMOTE:
(array([0, 1, 2]), array([10115, 12616,  6598]))


[I 2026-04-07 19:10:35,586] A new study created in memory with name: no-name-277b3d3a-47c0-48a4-9cb1-6b8b38974be6
[I 2026-04-07 19:10:35,620] Trial 0 finished with value: 0.7480383795989564 and parameters: {'alpha': 0.057543382035679325}. Best is trial 0 with value: 0.7480383795989564.
[I 2026-04-07 19:10:35,655] Trial 1 finished with value: 0.7496536038178437 and parameters: {'alpha': 0.0018489035576209627}. Best is trial 1 with value: 0.7496536038178437.
[I 2026-04-07 19:10:35,689] Trial 2 finished with value: 0.7496536038178437 and parameters: {'alpha': 0.0018123429493616399}. Best is trial 1 with value: 0.7496536038178437.
[I 2026-04-07 19:10:35,721] Trial 3 finished with value: 0.7496110205951734 and parameters: {'alpha': 0.007578771128007316}. Best is trial 1 with value: 0.7496536038178437.
[I 2026-04-07 19:10:35,757] Trial 4 finished with value: 0.7499525007533306 and parameters: {'alpha': 0.00022790280765260773}. Best is trial 4 with value: 0.7499525007533306.



🚀 TRAIN LABELS AFTER SMOTE:
(array([0, 1, 2]), array([12616, 12616, 12616]))


[I 2026-04-07 19:10:35,794] Trial 5 finished with value: 0.7498132227872932 and parameters: {'alpha': 0.0006405893757735349}. Best is trial 4 with value: 0.7499525007533306.
[I 2026-04-07 19:10:35,829] Trial 6 finished with value: 0.7470203349371575 and parameters: {'alpha': 0.4597286965912584}. Best is trial 4 with value: 0.7499525007533306.
[I 2026-04-07 19:10:35,864] Trial 7 finished with value: 0.7496536038178437 and parameters: {'alpha': 0.002906827411613982}. Best is trial 4 with value: 0.7499525007533306.
[I 2026-04-07 19:10:35,903] Trial 8 finished with value: 0.7471428583932519 and parameters: {'alpha': 0.14814243957359616}. Best is trial 4 with value: 0.7499525007533306.
[I 2026-04-07 19:10:35,937] Trial 9 finished with value: 0.7473825955374168 and parameters: {'alpha': 0.2986010432338614}. Best is trial 4 with value: 0.7499525007533306.
[I 2026-04-07 19:10:35,973] Trial 10 finished with value: 0.7502399679286714 and parameters: {'alpha': 0.00012783229208772982}. Best is tri


🏆 BEST PARAMS: {'alpha': 0.00012783229208772982}


2026/04/07 19:10:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/07 19:11:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run MultinomialNB_SMOTE_TFIDF_Bigram at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/9/runs/d1589b78136d42789dcafffe23ac1609
🧪 View experiment at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/9


In [27]:
best_params = {
    'alpha': 0.000127  # <-- apna best alpha daal yaha
}

model = MultinomialNB(
    alpha=best_params['alpha']
)

model.fit(X_train_res, y_train_res)

y_pred = model.predict(X_test)

from sklearn.metrics import f1_score

print("Test Macro F1:", f1_score(y_test, y_pred, average='macro'))
print("Test Weighted F1:", f1_score(y_test, y_pred, average='weighted'))

Test Macro F1: 0.7220286435483155
Test Weighted F1: 0.7372891733778674


In [28]:
import numpy as np

print("Unique predictions:", np.unique(y_pred))
print("Unique true labels:", np.unique(y_test))

Unique predictions: [0 1 2]
Unique true labels: [0 1 2]
